In [ ]:
import numpy as np
import pandas as pd
import re

resume = pd.read_csv('./data/resume.csv')
job = pd.read_csv('./data/postings.csv', engine='python')

In [6]:
job.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [7]:
job_sample = job.sample(2000)
job_sample.shape

(2000, 31)

In [8]:
def preprocess_text(text):
    """Preprocess text by converting to lowercase, removing special characters, and handling NaN."""
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

def combine_features(row):
    """Combine relevant features into a single string."""
    features = []
    for col in ['title', 'description', 'skills_desc']:
        if not pd.isnull(row[col]):
            features.append(f"{col.capitalize()}: {preprocess_text(row[col])}\n")
    return ' '.join(features)

job_sample['processed_text'] = job_sample.apply(combine_features, axis=1)

job_sample[['job_id', 'processed_text']].head()

,job_id,processed_text
73515,3902933153,Title: maintenance 2nd shift controls technic...
123777,3906262087,Title: project manager\n Description: title pr...
65043,3902365096,Title: senior finance manager customer servic...
102534,3905283617,Title: traveling surgical assistant\n Descript...
111900,3905399286,Title: supervisor retail\n Description: summar...


In [9]:
def preprocess_resume_text(row):
    """Preprocess resume text by converting to lowercase, removing special characters, and handling NaN."""
    text = row.get('Resume_str', '')
    if pd.isnull(text):
        return ""
    text = re.sub(r'[^\w\s,+./-]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    text = text.lower()

    return text

# Apply preprocessing
resume['processed_text'] = resume.apply(preprocess_resume_text, axis=1)

# Display the first 5 rows
resume[['ID', 'processed_text']].head()

,ID,processed_text
0,16852973,hr administrator/marketing associate hr admini...
1,22323967,"hr specialist, us hr operations summary versat..."
2,33176873,hr director summary over 20 years experience i...
3,27018550,"hr specialist summary dedicated, driven, and d..."
4,17812897,hr manager skill highlights hr skills hr depar...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer(norm="l2", max_features=5000)
resume_matrix = vectorizer.fit_transform(resume["processed_text"])
job_matrix = vectorizer.transform(job_sample["processed_text"])

similarity_scores = cosine_similarity(resume_matrix, job_matrix).round(3)
print(similarity_scores)

top_k = 5
top_k_indices = similarity_scores.argsort(axis=1)[:, -top_k:][:, ::-1]
print(top_k_indices)


recommended_jobs = [[job_sample.iloc[idx]["title"] for idx in job_indices] for job_indices in top_k_indices]
resume["recommended_jobs"] = recommended_jobs
resume["similarity_scores"] = [similarity_scores[i, job_indices].tolist() for i, job_indices in enumerate(top_k_indices)]
resume

[[0.202 0.206 0.308 ... 0.271 0.18  0.256]
 [0.22  0.221 0.295 ... 0.298 0.202 0.265]
 [0.27  0.285 0.353 ... 0.356 0.264 0.316]
 ...
 [0.154 0.109 0.164 ... 0.151 0.108 0.143]
 [0.19  0.185 0.27  ... 0.261 0.222 0.236]
 [0.209 0.162 0.239 ... 0.271 0.174 0.244]]
[[1871  888  788 1512 1630]
 [ 909 1511  888 1871  977]
 [ 725 1630 1919  788 1556]
 ...
 [1522  406  891 1575  967]
 [1630 1528 1919 1654  171]
 [1522  891 1006  206 1630]]


,ID,Resume_str,Resume_html,Category,processed_text,recommended_jobs,similarity_scores
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administrator/marketing associate hr admini...,"[Director of Marketing Operations, Senior Dire...","[0.428, 0.422, 0.415, 0.412, 0.408]"
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,"hr specialist, us hr operations summary versat...","[Marketing Director, Events Operation Speciali...","[0.48, 0.476, 0.464, 0.459, 0.446]"
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary over 20 years experience i...,"[Human Resources Manager, Director of Member E...","[0.535, 0.519, 0.515, 0.508, 0.491]"
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,"hr specialist summary dedicated, driven, and d...","[Director of Member Experience , Logistics Man...","[0.449, 0.402, 0.399, 0.397, 0.391]"
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlights hr skills hr depar...,"[Human Resources Manager, People Experience Ex...","[0.595, 0.534, 0.523, 0.517, 0.502]"
...,...,...,...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,rank sgt/e-5 non- commissioned officer in char...,"[Logistics Manager, Director of Member Experie...","[0.398, 0.379, 0.373, 0.369, 0.368]"
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION,"government relations, communications and organ...","[Operations Manager, Director of Member Experi...","[0.503, 0.476, 0.47, 0.449, 0.445]"
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,geek squad agent professional profile it suppo...,"[Shipping and Receiving Clerk, Warehouse Manag...","[0.332, 0.32, 0.294, 0.292, 0.28]"
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,program director / office manager summary high...,"[Director of Member Experience , Executive Ass...","[0.403, 0.388, 0.383, 0.372, 0.366]"


In [11]:
print(f'Mean: {np.mean(similarity_scores)}, Median: {np.median(similarity_scores)}, Max: {np.max(similarity_scores)}, Min: {np.min(similarity_scores)}')

Mean: 0.22243295933977453, Median: 0.221, Max: 0.715, Min: 0.0


In [12]:
import numpy as np

# Flatten all top-5 similarity scores across all resumes
all_top5_scores = [score for scores_list in resume["similarity_scores"] for score in scores_list]

# Compute the average
average_top5_similarity = np.mean(all_top5_scores)

print(f"Average Top-5 Similarity Score: {average_top5_similarity:.4f}")

Average Top-5 Similarity Score: 0.4169
